# Phase 3 baseline: Colab pipeline

Runs top to bottom, no cell edits, on a **T4 GPU runtime** (Runtime > Change runtime type).

This machine has no GPU and a slow connection (~1.25 MB/s to the HF CDN, `results/throughput_laptop.json`), so everything below runs here instead. Git is the bridge: this notebook clones the **public** repo anonymously (no token needed), and at the end copies results back to Drive for a laptop to commit -- Colab never pushes (see HANDOFF.md "Workflow").

One deviation from the plan as originally written: PLAN.md sketches "download" and "normalize" as two separate steps. In this codebase they are not separable -- `scripts/download_data.py` calls the canonical decode path (`src/data/normalize.py`) inline on every row as it streams the parquet, and never writes a raw file to disk (Phase 2 design: "nothing raw is ever written"). So Cell 2 below does both at once; there is no standalone normalization pass to run afterward.

## Cell 1 -- mount Drive, clone, install, verify the bridge

Runs `scripts/bench_throughput.py` last, so a broken clone/install/GPU fails here in well under a minute rather than 10+ minutes into a download.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/anniketkumar/aigc-detect.git"

REPO_DIR = "/content/repo"
DRIVE_ROOT = "/content/drive/MyDrive/aigc"

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/features", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Public repo -- anonymous clone, no auth, no token.
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -q -r requirements.txt

fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo


In [3]:
import torch

has_gpu = torch.cuda.is_available()
print("CUDA available:", has_gpu)
print("GPU:", torch.cuda.get_device_name(0) if has_gpu else "NONE")
assert has_gpu, (
    "No GPU visible. Runtime > Change runtime type > T4 GPU, then Runtime > "
    "Restart session and rerun from the top. A CPU runtime will make cell 4 "
    "(CLIP feature caching) take hours instead of minutes."
)

CUDA available: True
GPU: Tesla T4


In [4]:
# The bridge check. If this fails or reports laptop-like throughput, stop and
# fix the environment before spending time on cell 2's download.
!python -m scripts.bench_throughput --out results/throughput_colab.json

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.13.15",
  "cpu_count": 2,
  "in_colab": true,
  "in_kaggle": true,
  "torch": "2.13.0+cu130",
  "cuda": true,
  "gpu": "Tesla T4",
  "gpu_mem_gb": 15.6
}

[cdn] SID_Set ...
            40.81 MB/s  (147 GB/h)   {'file_bytes': 477663216, 'chunk_mb': 50.3, 'reps': 3, 'mbps_median': 40.81, 'mbps_all': [40.81, 37.23, 41.01]}
[parquet] SID_Set ...
            30.88 MB/s  (111 GB/h)   {'footer_read_s': 0.78, 'row_groups': 1, 'rows': 100, 'uncompressed_mb': 55.9, 'seconds': 1.81, 'mbps': 30.88, 'images_per_s': 55.2}

[cdn] GenImage_BigGAN ...
            41.74 MB/s  (150 GB/h)   {'file_bytes': 652468032, 'chunk_mb': 50.3, 'reps': 3, 'mbps_median': 41.74, 'mbps_all': [41.74, 45.06, 38.92]}
[parquet] GenImage_BigGAN ...
              8.14 MB/s  (29 GB/h)   {'footer_read_s': 0.68, 'row_groups': 1, 'rows': 100, 'uncompressed_mb': 3.3, 'seconds': 0.41, 'mbps': 8.14, 'images_per_s': 245.6}

[disk] /tmp/aigc_bench ...
      {'path

## Cell 2 -- download + normalize (one step, see the note above), build manifests

Uses the default per-source quotas in `src/data/sources.py`: ~18.2k images, 4 training generators + 3 held-out generators (MidJourney, Gemini, FLUX.1-dev) + 3 real sources. Images land in `/content` (ephemeral) -- nothing here goes to Drive, per the Workflow note (free tier is 15 GB and images get re-pulled every session anyway).

In [5]:
!python -m scripts.download_data --out data/corpus

source                   lab  quota  MB/img transfer GB  disk GB
SDXL                       1   2000   1.392        3.28    0.190
Mobius                     1   2000   1.527        3.59    0.190
RealVisXL-V4.0             1   2000   1.422        3.35    0.190
Aura                       1   2000   0.505        1.19    0.190
MidJourney                 1   2000   0.751        1.77    0.190
Gemini-nano-banana         1   2000   1.576        3.71    0.190
FLUX.1-dev                 1    800   0.566        0.53    0.076
OpenImagesV7               0   1800   0.566        1.20    0.171
Megalith-Flickr            0   1800   0.194        0.41    0.171
Unsplash                   0   1800   0.130        0.28    0.171

  images      18,200
  transfer    19.31 GB  (network)
  on disk     1.73 GB  (normalized, kept)
  peak disk   1.93 GB

config 62e4f79471ec; ledger has 2879 rows

[1/10] OpenImagesV7 (saberzl/SID_Set default/validation)
  OpenImagesV7: 1800/1800 already done
[2/10] Megalith-Flickr (b

In [6]:
!python -m src.data.manifest --ledger data/corpus/ledger.csv --out data/manifests
!python -m scripts.make_data_stats

  hashed 2000/18200
  hashed 4000/18200
  hashed 6000/18200
  hashed 8000/18200
  hashed 10000/18200
  hashed 12000/18200
  hashed 14000/18200
  hashed 16000/18200
  hashed 18000/18200
{
  "n_images": 18200,
  "splits": {
    "train": 9380,
    "val": 2010,
    "test": 6810
  },
  "n_clusters": 18198,
  "n_dup_pairs": 2,
  "n_blocked": 2,
  "blocklist_empty": false,
  "blocklist_sources": [
    "COCO val2017: 5000 images, 5000 sha256, 5000 phash"
  ],
  "blocklist_gaps": [
    "DALL-E Advanced (8,843 imgs): no public standalone distribution -- it ships only inside WildFake's ModelScope zips (~700 GB), which is not a feasible download here. Not hashed. Mitigated by the source-registry denylist asserted in tests/test_manifest.py::test_no_dalle_derived_source_in_the_registry."
  ],
  "holdout_generators": [
    "FLUX.1-dev",
    "Gemini-nano-banana",
    "MidJourney"
  ],
  "by_split_label": {
    "0": {
      "test": 810,
      "train": 3780,
      "val": 810
    },
    "1": {
      "tes

## Gate -- the two non-negotiable guards, plus the rest of the suite

`tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant` only *engages* once `data/corpus/ledger.csv` exists (HANDOFF.md's documented known gap, `tests/test_manifest.py:456`) -- which is exactly now, for the first time this session. This is the one place in the whole pipeline where the real WildFake/DALL·E blocklist is actually exercised against real data. **Do not skip this cell or continue past a failure.**

In [7]:
import subprocess

# Colab does not capture a child process's fd 1, so the previous run's gate
# recorded nothing but its own success line -- exit 0 proved the print ran, not
# that the guards engaged. Capture the output, show the tail, and pass -rs so
# skips are visible: both guards are skipif-gated on data/corpus/ledger.csv.
result = subprocess.run(
    ["python", "-m", "pytest", "-q", "-rs"], capture_output=True, text=True
)
print(result.stdout[-4000:])
if result.stderr.strip():
    print("--- stderr (tail) ---")
    print(result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit(
        "pytest failed. Do not continue past this cell -- this is the run where "
        "test_manifest.py's WildFake content-hash guard and the DALL·E "
        "source-registry denylist test actually see real data (HANDOFF.md "
        "'Non-negotiable'). A failure here is a disqualification risk, not a "
        "flaky test to retry past."
    )

# Exit 0 alone is not evidence: a skipif-skipped guard also exits 0. Name the
# two non-negotiables and require PASSED, so the claim below is witnessed.
GUARDS = [
    "tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant",
    "tests/test_manifest.py::test_no_dalle_derived_source_in_the_registry",
]
witness = subprocess.run(
    ["python", "-m", "pytest", "-v", "-rs", *GUARDS], capture_output=True, text=True
)
print(witness.stdout[-2500:])
n_passed = witness.stdout.count("PASSED")
if witness.returncode != 0 or n_passed != len(GUARDS):
    raise SystemExit(
        f"the WildFake/DALL·E guards did not both PASS ({n_passed}/{len(GUARDS)}). "
        "If they SKIPPED, data/corpus/ledger.csv is missing and the "
        "non-negotiable check never ran -- HANDOFF.md, disqualification risk."
    )
print(
    f"pytest green, and both real-corpus guards PASSED "
    f"({n_passed}/{len(GUARDS)}) -- witnessed by name, not assumed"
)

........................................................................ [ 27%]
........................................................................ [ 54%]
........................................................................ [ 82%]
..............................................                           [100%]
262 passed in 329.04s (0:05:29)

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: langsmith-0.11.1, typeguard-4.6.0, anyio-4.14.2
collecting ... collected 2 items

tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant PASSED [ 50%]
tests/test_manifest.py::test_no_dalle_derived_source_in_the_registry PASSED [100%]

======================== 2 passed in 121.25s (0:02:01) =========================

pytest green, and both real-corpus guards PASSED (2/2) -- witnessed by name, not ass

## Cell 4 -- cache frozen CLIP ViT-B/16 features to Drive

One pass per split. After this, every later head experiment (Phase 3 baseline, and any Phase 4+ head swap) reads three small `.npy`/`.json` files and needs no GPU.

In [8]:
FEATURES_DIR = f"{DRIVE_ROOT}/features"

for split in ("train", "val", "test"):
    !python -m scripts.cache_features \
        --manifest data/manifests/{split}.csv \
        --out {FEATURES_DIR}/{split} \
        --device cuda --batch-size 256

loading ViT-B-16-quickgelu (openai) on cuda ...


  backbone ready in 12.1s, embed_dim=512

{
  "manifest": "data/manifests/train.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "embed_dim": 512,
  "n_images": 9380,
  "n_failed": 0,
  "n_real": 3780,
  "n_ai": 5600,
  "elapsed_s": 193.9
}
-> /content/drive/MyDrive/aigc/features/train
loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 8.1s, embed_dim=512

{
  "manifest": "data/manifests/val.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "embed_dim": 512,
  "n_images": 2010,
  "n_failed": 0,
  "n_real": 810,
  "n_ai": 1200,
  "elapsed_s": 48.5
}
-> /content/drive/MyDrive/aigc/features/val
loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 8.5s, embed_dim=512

{
  "manifest": "data/manifests/test.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "embed_dim": 512,
  "n_images": 6810,
  "n_failed": 0,
  "n_real": 810,
  "n_ai": 6000,
  "elapse

## Cell 5 -- train the baseline head

Linear head, BCE, no augmentation -- the deliberate control (HANDOFF.md), now trained to convergence (300 epochs, early stopping patience 30 on val AUROC). Checkpoint goes straight to Drive, not to `/content`, because free Colab disconnects on idle and this is cheap to lose only once.

In [9]:
CKPT = f"{DRIVE_ROOT}/checkpoints/baseline.pt"

# 300 epochs with early stopping (patience 30 on val AUROC), not a flat 50: the
# first run's val AUROC was still climbing at epoch 49, so best-val selection
# returned the final epoch and never selected anything. An unconverged control
# inflates Phase 4's apparent augmentation gain. Costs ~4 seconds.
!python -m src.train \
    --features-dir {FEATURES_DIR} \
    --train-split train --val-split val \
    --out {CKPT} \
    --epochs 300 --patience 30 --device cuda

epoch   0  loss 0.6729  train_auroc 0.9292  val_auroc 0.9313
epoch   1  loss 0.6405  train_auroc 0.9495  val_auroc 0.9518
epoch   2  loss 0.6150  train_auroc 0.9568  val_auroc 0.9593
epoch   3  loss 0.5932  train_auroc 0.9601  val_auroc 0.9625
epoch   4  loss 0.5736  train_auroc 0.9623  val_auroc 0.9645
epoch   5  loss 0.5556  train_auroc 0.9637  val_auroc 0.9661
epoch   6  loss 0.5386  train_auroc 0.9646  val_auroc 0.9670
epoch   7  loss 0.5229  train_auroc 0.9653  val_auroc 0.9678
epoch   8  loss 0.5081  train_auroc 0.9660  val_auroc 0.9685
epoch   9  loss 0.4942  train_auroc 0.9666  val_auroc 0.9691
epoch  10  loss 0.4811  train_auroc 0.9671  val_auroc 0.9696
epoch  11  loss 0.4689  train_auroc 0.9676  val_auroc 0.9701
epoch  12  loss 0.4574  train_auroc 0.9680  val_auroc 0.9705
epoch  13  loss 0.4465  train_auroc 0.9685  val_auroc 0.9710
epoch  14  loss 0.4362  train_auroc 0.9689  val_auroc 0.9713
epoch  15  loss 0.4266  train_auroc 0.9693  val_auroc 0.9718
epoch  16  loss 0.4174  

## Cell 6 -- run the Phase 1 harness on the held-out test split

`--split test` includes both the transform grid *and* the three held-out generators (MidJourney, Gemini, FLUX.1-dev), since `src/data/manifest.py` routes them there in full.

In [14]:
!python -m src.evaluate \
    --model clip_linear --ckpt {CKPT} \
    --split test --out results/baseline/ \
    --device cuda

model=clip_linear(/content/drive/MyDrive/aigc/checkpoints/baseline.pt)  images=6810 (real=810, ai=6000)  cells=19  seed=0
AUROC null SD at this sample size: 0.0108 (a chance-level model should land within ~0.032 of 0.500)


clean AUROC        0.9810
mean transformed   0.9710   (family-balanced over 7 families)
robustness_gap     +0.0099   (lower is better)
worst_case         0.9484   (composed_resize0.25+blur0.5+jpeg30)
  flat mean        0.9689   gap +0.0120   (§3.2 literal, secondary)

wrote results/baseline/grid.csv and results/baseline/report.md in 3337.43s


In [15]:
import json

summary = json.loads(open("results/baseline/summary.json").read())["summary"]
clean_auroc = summary["clean_auroc"]
print(f"clean AUROC: {clean_auroc:.4f}")

if clean_auroc > 0.99:
    print(
        "\n*** STOP AND FLAG THIS ***\n"
        "HANDOFF.md: clean AUROC above 0.99 is far more likely to be a "
        "surviving leak than a good model, given what the Phase 2 audit found "
        "in raw SID_Set. Do not treat this as a good result -- check the "
        "held-out generator breakdown and re-run the Phase 2 normalization "
        "audit (tests/test_normalization_audit.py) before trusting it."
    )
elif not (0.85 <= clean_auroc <= 0.95):
    print(
        f"note: clean AUROC {clean_auroc:.4f} is outside the expected "
        "0.85-0.95 band (HANDOFF.md). Not necessarily wrong -- just worth a "
        "second look before reporting it as the baseline."
    )

clean AUROC: 0.9810
note: clean AUROC 0.9810 is outside the expected 0.85-0.95 band (HANDOFF.md). Not necessarily wrong -- just worth a second look before reporting it as the baseline.


## Cell 6b -- aesthetic probe: how much of the number is "looks artistic"?

Two pixel features (`hf_energy`, `flat_frac`), logistic regression, no CLIP. The first run had unseen MidJourney (0.9894) beating every trained-on generator while unseen FLUX sat at 0.433 TPR@1% -- the signature of a style detector, not a forensic one. If two texture statistics reproduce that ordering, the headline is substantially aesthetics.

Runs after cell 6 on purpose: it reads `results/baseline/scores.csv` to put the CLIP baseline's per-generator AUROC beside its own. CPU only, ~3-5 min.

In [16]:
# --verify-against-audit first checks these two features match
# scripts.audit_leakage.content_features exactly, so the probe is the Phase 2
# measure and not a second invented one.
!python -m scripts.aesthetic_probe \
    --manifests data/manifests \
    --out results/baseline \
    --verify-against-audit 20


verified 20 image(s): both feature paths agree exactly
measuring 9380 train + 6810 test images on hf_energy, flat_frac ...


{
  "overall": {
    "auroc": 0.6085399176954733,
    "tpr_at_fpr1": 0.03216666666666667
  },
  "spearman_vs_clip": {
    "rho": -0.14285714285714288,
    "p": 0.7599453002180929,
    "n_generators": 7
  },
  "elapsed_s": 158.0
}
-> results/baseline/aesthetic_probe.md


## Cell 7 -- copy results back to Drive (no git push from Colab)

A laptop pulls this folder from Drive and commits it -- Colab never pushes, which removes an auth failure mode deliberately (HANDOFF.md "Workflow").

In [17]:
import shutil

shutil.copytree("results/baseline", f"{DRIVE_ROOT}/results/baseline", dirs_exist_ok=True)
shutil.copy("results/data_stats.md", f"{DRIVE_ROOT}/results/data_stats.md")
print(f"copied to {DRIVE_ROOT}/results/baseline and {DRIVE_ROOT}/results/data_stats.md")
print("On the laptop: pull these from Drive into results/baseline/ and results/data_stats.md, then git add + commit + push.")

copied to /content/drive/MyDrive/aigc/results/baseline and /content/drive/MyDrive/aigc/results/data_stats.md
On the laptop: pull these from Drive into results/baseline/ and results/data_stats.md, then git add + commit + push.
